In [67]:
from pathlib import Path

import altair as alt
import pandas as pd
from IPython.display import display

DATA_PATH = Path("W168_Apr25/W168_Apr25/ATP W168.sav")

In [68]:
import pyreadstat
import pandas as pd

df, metadata = pyreadstat.read_sav(DATA_PATH)

# 1. Replace coded values with their value labels
for col, labels in metadata.variable_value_labels.items():
    if col in df.columns:
        df[col] = df[col].replace(labels)

# 2. Rename columns using variable labels
label_map = metadata.column_names_to_labels

df = df.rename(columns={
    col: label_map.get(col, col)
    for col in df.columns
})

print(df.head())

   QKEY. Unique identifier INTERVIEW_START_W168. Interview start time  \
0                 100314.0                       16-Apr-2025 10:54:09   
1                 100363.0                       15-Apr-2025 19:46:25   
2                 100637.0                       17-Apr-2025 18:11:51   
3                 100803.0                       16-Apr-2025 17:39:30   
4                 101437.0                       16-Apr-2025 15:56:54   

  INTERVIEW_END_W168. Interview end time  \
0                   16-Apr-2025 11:10:27   
1                   15-Apr-2025 19:56:06   
2                   17-Apr-2025 18:16:30   
3                   16-Apr-2025 17:51:05   
4                   16-Apr-2025 16:18:38   

  DEVICE_TYPE_W168. The device used in the latest access of the survey link  \
0                                             Mobile                          
1                                             Mobile                          
2                                            Desktop       

In [69]:
df.columns

Index(['QKEY. Unique identifier', 'INTERVIEW_START_W168. Interview start time',
       'INTERVIEW_END_W168. Interview end time',
       'DEVICE_TYPE_W168. The device used in the latest access of the survey link',
       'SVYMODE_W168. Mode of survey', 'LANG_W168. Language of interview',
       'FORM_W168. Flag to randomly assign panelists to one of two forms (Form 1, Form 2) and weight within form ATP source: Randomly assigned for each survey.',
       'SNSUSE_W168. Do you ever use social media sites like Facebook, X (formerly Twitter), or Instagram?',
       'ONLTSK_a_W168. Buy things online // Do you ever do each of the following?',
       'ONLTSK_b_W168. Use online banking sites or apps // Do you ever do each of the following?',
       ...
       'Party closeness among leaners', 'Party summary',
       'Combining ideology and party identification', 'Voter registration',
       'Family income', 'Ideology', 'Frequency of internet use',
       'Volunteerism status', 'Income tier 3-way'

In [70]:
print(df.columns.tolist())

['QKEY. Unique identifier', 'INTERVIEW_START_W168. Interview start time', 'INTERVIEW_END_W168. Interview end time', 'DEVICE_TYPE_W168. The device used in the latest access of the survey link', 'SVYMODE_W168. Mode of survey', 'LANG_W168. Language of interview', 'FORM_W168. Flag to randomly assign panelists to one of two forms (Form 1, Form 2) and weight within form ATP source: Randomly assigned for each survey.', 'SNSUSE_W168. Do you ever use social media sites like Facebook, X (formerly Twitter), or Instagram?', 'ONLTSK_a_W168. Buy things online // Do you ever do each of the following?', 'ONLTSK_b_W168. Use online banking sites or apps // Do you ever do each of the following?', 'ONLTSK_c_W168. Send or receive emails // Do you ever do each of the following?', 'ONLTSK_d_W168. Send or receive text messages // Do you ever do each of the following?', 'STREAM1_W168. Do you ever watch programming on streaming services, such as Netflix, Disney+ or Amazon Prime Video?', 'CABLETV1_W168. Do you c

In [71]:
# Allow larger datasets in Altair
alt.data_transformers.disable_max_rows()

# Survey variables to exclude from analysis
exclude_cols = [
    "QKEY. Unique identifier",
    "INTERVIEW_START_W168. Interview start time",
    "INTERVIEW_END_W168. Interview end time"
]

# Demographic variables for comparison
demographic_cols = [
    "Age category",
    "Gender",
    "Education level category",
    "Income tier 3-way",
    "Party",
    "Census region",
    " Self-identified Urban, Suburban, Rural (USR)"
]

# Use the existing human-readable question labels as field names
print(df.shape)
print(df.dtypes.value_counts())

(9397, 151)
object     147
float64      2
str          2
Name: count, dtype: int64


In [72]:
weight_col = "Wave 168 weight"

def weighted_counts(data, group_cols, weight_col=weight_col):
    """
    Collapse to weighted counts (effectively weighted frequencies)
    for one or more grouping columns.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    return (
        data.groupby(group_cols, dropna=False)[weight_col]
        .sum()
        .reset_index(name="Weighted_n")
    )

def weighted_share(data, group_cols, within_cols, weight_col=weight_col):
    """
    Weighted share of each group_cols combination, normalized within within_cols.
    e.g., weighted_share(df, ['Party', confidence_col], ['Party'])
    gives the weighted % for each confidence level within each party.
    """
    if isinstance(within_cols, str):
        within_cols = [within_cols]
    summary = weighted_counts(data, group_cols, weight_col)
    summary["Weighted_pct"] = (
        summary["Weighted_n"]
        / summary.groupby(within_cols)["Weighted_n"].transform("sum")
        * 100
    )
    return summary

In [73]:
def response_distribution(data, column, title=None):
    plot_df = (
        data[column]
        .fillna("Missing")
        .astype(str)
        .value_counts()
        .rename_axis("Response")
        .reset_index(name="Respondents")
    )

    chart = (
        alt.Chart(plot_df)
        .mark_bar()
        .encode(
            x=alt.X(
                "Respondents:Q",
                title="Number of respondents"
            ),
            y=alt.Y(
                "Response:N",
                sort="-x",
                title=None
            ),
            tooltip=["Response:N", "Respondents:Q"]
        )
        .properties(
            title=title or column,
            width=450,
            height=300
        )
    )

    return chart

In [74]:
response_distribution(
    df,
    "SNSUSE_W168. Do you ever use social media sites like Facebook, X (formerly Twitter), or Instagram?"
)

alt.Chart(...)

In [75]:
confidence_col = "journalist_confidence"

df = df.rename(columns={
    "CONF_b_W168. How much confidence, if any, do you have in journalists to act in the best interests of the public?": confidence_col
})

dont_know_refused = "Don't know/Refused/Web blank"

df_confidence = df[
    df[confidence_col].notna()
    & (df[confidence_col] != dont_know_refused)
].copy()

party_confidence_summary = weighted_share(
    df_confidence,
    ["Party", confidence_col],
    ["Party"]
)

party_order = [
    "Refused", "Something else", "Democrat",
    "Independent", "Republican"
]



confidence_order = [
    "Don't know/Refused/Web blank",
    "No confidence at all",
    "Not too much confidence",
    "A fair amount of confidence",
    "A great deal of confidence"
]


party_chart = (
    alt.Chart(party_confidence_summary)
    .mark_bar()
    .encode(
        x=alt.X(
            "Party:N",
            title="Political party",
            sort=party_order
        ),
        y=alt.Y(
            "Weighted_n:Q",
            title="Number of respondents"
        ),
        color=alt.Color(
            f"{confidence_col}:O",
            title="Confidence in journalists",
            sort=confidence_order
        ),
        xOffset=alt.XOffset(
            f"{confidence_col}:N",
            sort=confidence_order
        ),
        tooltip=[
            alt.Tooltip("Party:N"),
            alt.Tooltip(f"{confidence_col}:N", title="Confidence"),
            alt.Tooltip("Weighted_n:Q")
        ]
    )
    .properties(
        title="Confidence in Journalists by Political Party",
        width=550,
        height=350
    )
)

party_chart

alt.Chart(...)

In [76]:
party_confidence_summary

,Party,journalist_confidence,Weighted_n,Weighted_pct
0,Democrat,A fair amount of confidence,1440.375146,55.937042
1,Democrat,A great deal of confidence,275.705175,10.707024
2,Democrat,No confidence at all,105.969564,4.115333
3,Democrat,Not too much confidence,752.943545,29.240601
4,Independent,A fair amount of confidence,1062.300539,39.715917
5,Independent,A great deal of confidence,103.741769,3.878563
6,Independent,No confidence at all,389.901197,14.577121
7,Independent,Not too much confidence,1118.804096,41.828399
8,Refused,A fair amount of confidence,70.936838,46.078533
9,Refused,A great deal of confidence,11.582163,7.523440


In [77]:
confidence_rank = {
    confidence: i for i, confidence in enumerate(confidence_order)
}

party_confidence_summary["Confidence order"] = (
    party_confidence_summary[confidence_col].map(confidence_rank)
)

party_chart = (
    alt.Chart(party_confidence_summary)
    .mark_bar()
    .encode(
        x=alt.X(
            "Party:N",
            title="Political party",
            sort=party_order,
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "Weighted_n:Q",
            title="Percentage of respondents",
            stack="normalize",
            axis=alt.Axis(format="%")
        ),
        color=alt.Color(
            f"{confidence_col}:O",
            title="Confidence in journalists",
            sort =confidence_order
        ),
        order=alt.Order(
            "Confidence order:Q",
            sort="ascending"),
        tooltip=[
            alt.Tooltip(f"{confidence_col}:O", title="Confidence in journalists"),
            alt.Tooltip("Party:N", title="Party affiliation"),
            alt.Tooltip("Weighted_pct:Q", title="Respondents")
        ]
    )
    .properties(
        title="Journalist Confidence Within Each Political Party",
        width=500,
        height=350
    )
)

party_chart

alt.Chart(...)

In [78]:
import pandas as pd
import altair as alt

gender_col = "Gender"
news_col = "FOLNEWS_W168. Would you say you follow the news..."

# Keep respondents with valid gender and news responses
news_df = df[[gender_col, news_col]].dropna().copy()

# Calculate the share within each gender for every response category
news_long = (
    news_df
    .groupby([news_col, gender_col])
    .size()
    .reset_index(name="n")
)

news_long["rate"] = (
    news_long["n"] /
    news_long.groupby(gender_col)["n"].transform("sum")
)

news_long = news_long.rename(columns={
    news_col: "News response",
    gender_col: "group"
})

print(news_long.head())
print(news_long["group"].unique())
print(news_long["News response"].unique())

                  News response              group     n      rate
0       All or most of the time              A man  2379  0.570640
1       All or most of the time            A woman  2136  0.414838
2       All or most of the time  In some other way    15  0.306122
3       All or most of the time            Refused    10  0.333333
4  Don't know/Refused/Web blank              A man    24  0.005757
<StringArray>
['A man', 'A woman', 'In some other way', 'Refused']
Length: 4, dtype: str
<StringArray>
[     'All or most of the time', 'Don't know/Refused/Web blank',
                  'Hardly ever',            'Only now and then',
             'Some of the time']
Length: 5, dtype: str


In [79]:
gender_order = ["A man", "A woman"]

gender_colors = alt.Scale(
    domain=gender_order,
    range=["#4C78A8", "#E45756"]
)

# Order responses by overall frequency, or replace with a logical survey order
news_order = (
    news_df[news_col]
    .value_counts()
    .index
    .tolist()
)

wide = (
    news_long
    .pivot(index="News response", columns="group", values="rate")
    .reset_index()
)

# Ensure both gender columns exist before drawing connecting lines
wide = wide.dropna(subset=gender_order)

y = alt.Y(
    "News response:N",
    title=None,
    sort=news_order,
    axis=alt.Axis(labelLimit=250)
)

x = alt.X(
    "rate:Q",
    title="Share within gender",
    scale=alt.Scale(domain=[0, 1]),
    axis=alt.Axis(format="%", tickCount=5)
)

rules = (
    alt.Chart(wide)
    .mark_rule(color="gray", strokeWidth=2)
    .encode(
        y=y,
        x=alt.X(
            f"{gender_order[0]}:Q",
            title="Share within gender",
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(format="%", tickCount=5)
        ),
        x2=f"{gender_order[1]}:Q"
    )
)

dots = (
    alt.Chart(news_long)
    .mark_circle(size=100)
    .encode(
        y=y,
        x=x,
        color=alt.Color(
            "group:N",
            title="Gender",
            scale=gender_colors
        ),
        tooltip=[
            alt.Tooltip("News response:N", title="News-following response"),
            alt.Tooltip("group:N", title="Gender"),
            alt.Tooltip("rate:Q", title="Share", format=".1%"),
            alt.Tooltip("n:Q", title="Respondents")
        ]
    )
)

news_dumbbell = (
    (rules + dots)
    .properties(
        title="News-Following Habits by Gender",
        width=500,
        height=300
    )
)

news_dumbbell

alt.LayerChart(...)

In [80]:
platform_cols = {
    "Text messages / phone calls": "OSCAM_PLAT_a_W168. Text messages and phone calls // How much of a problem do you think online scams and attacks are on each of the following?",
    "Email": "OSCAM_PLAT_b_W168. Emails // How much of a problem do you think online scams and attacks are on each of the following?",
    "Social media": "OSCAM_PLAT_c_W168. Social media sites or apps // How much of a problem do you think online scams and attacks are on each of the following?",
    "Shopping sites": "OSCAM_PLAT_d_W168. Shopping sites or apps // How much of a problem do you think online scams and attacks are on each of the following?",
    "Banking sites": "OSCAM_PLAT_e_W168. Banking sites or apps // How much of a problem do you think online scams and attacks are on each of the following?",
    "Payment apps": "OSCAM_PLAT_f_W168. Payment sites or apps, such as Venmo, PayPal or Cash App // How much of a problem do you think online scams and attacks are on each of the following?",
    "Cryptocurrency": "OSCAM_PLAT_g_W168. Cryptocurrency sites or apps // How much of a problem do you think online scams and attacks are on each of the following?",
    "Dating sites": "OSCAM_PLAT_h_W168. Dating sites or apps // How much of a problem do you think online scams and attacks are on each of the following?"
}

In [81]:
# Use the platform_cols dictionary from earlier
platform_long = (
    df[list(platform_cols.values())]
    .rename(columns={v: k for k, v in platform_cols.items()})
    .melt(var_name="Platform", value_name="Response")
    .dropna(subset=["Response"])
)

# Standardize numeric response codes or existing text labels
response_map = {
    "1": "Major problem",
    "1.0": "Major problem",
    "2": "Minor problem",
    "2.0": "Minor problem",
    "3": "Not a problem",
    "3.0": "Not a problem",
    "4": "Not sure",
    "4.0": "Not sure",
    "99": "Refused/Web blank",
    "99.0": "Refused/Web blank"
}

platform_long["Response"] = (
    platform_long["Response"]
    .astype(str)
    .str.strip()
    .replace(response_map)
)

# Keep the three categories separate
response_order = [
    "Major problem",
    "Minor problem",
    "Not a problem"
]

comparison_data = platform_long[
    platform_long["Response"].isin(response_order)
]

# Count responses by platform and category
comparison = (
    comparison_data
    .groupby(["Platform", "Response"])
    .size()
    .reset_index(name="Count")
)

# Calculate percentages for each platform
comparison["Total responses"] = (
    comparison.groupby("Platform")["Count"].transform("sum")
)

comparison["Percentage"] = (
    comparison["Count"] / comparison["Total responses"] * 100
)

comparison

,Platform,Response,Count,Total responses,Percentage
0,Banking sites,Major problem,2068,4055,50.998767
1,Banking sites,Minor problem,1653,4055,40.764488
2,Banking sites,Not a problem,334,4055,8.236745
3,Cryptocurrency,Major problem,1805,2537,71.147024
4,Cryptocurrency,Minor problem,672,2537,26.487978
5,Cryptocurrency,Not a problem,60,2537,2.364998
6,Dating sites,Major problem,1730,2676,64.648729
7,Dating sites,Minor problem,870,2676,32.511211
8,Dating sites,Not a problem,76,2676,2.840060
9,Email,Major problem,6301,8928,70.575717


In [82]:
chart = (
    alt.Chart(comparison)
    .mark_bar()
    .encode(
        x=alt.X(
            "Platform:N",
            title=None,
            sort=alt.EncodingSortField(
                field="Percentage",
                op="max",
                order="descending"
            ),
            axis=alt.Axis(labelAngle=-35, labelLimit=180)
        ),
        xOffset=alt.XOffset(
            "Response:N",
            sort=response_order
        ),
        y=alt.Y(
            "Percentage:Q",
            title="Percentage of responses",
            scale=alt.Scale(domain=[0, 100])
        ),
        color=alt.Color(
            "Response:N",
            title="Scam concern",
            sort=response_order
        ),
        tooltip=[
            alt.Tooltip("Platform:N"),
            alt.Tooltip("Response:N"),
            alt.Tooltip("Count:Q", title="Respondents"),
            alt.Tooltip("Percentage:Q", format=".1f")
        ]
    )
    .properties(
        title="Perceived Online Scam Problems by Platform",
        width=600,
        height=350
    )
)

chart

alt.Chart(...)

In [83]:


young_col = (
    "OSCAM_VULN_a_W168. Adults ages 30 and under // "
    "How likely do you think it is that each of the following "
    "would fall victim to an online scam or attack?"
)

older_col = (
    "OSCAM_VULN_b_W168. Adults ages 65 and older // "
    "How likely do you think it is that each of the following "
    "would fall victim to an online scam or attack?"
)

age_cols = {
    "Adults 30 and under": young_col,
    "Adults 65 and older": older_col
}

# Reshape into long format
vuln_long = (
    df[list(age_cols.values())]
    .rename(columns={v: k for k, v in age_cols.items()})
    .melt(var_name="Age group", value_name="Response")
    .dropna(subset=["Response"])
)

# Convert numeric response codes to labels if necessary
response_map = {
    "1": "Extremely likely",
    "1.0": "Extremely likely",
    "2": "Very likely",
    "2.0": "Very likely",
    "3": "Somewhat likely",
    "3.0": "Somewhat likely",
    "4": "Not too likely",
    "4.0": "Not too likely",
    "5": "Not at all likely",
    "5.0": "Not at all likely",
    "99": "Don't know/Refused/Web blank",
    "99.0": "Don't know/Refused/Web blank"
}

vuln_long["Response"] = (
    vuln_long["Response"]
    .astype(str)
    .str.strip()
    .replace(response_map)
)

# Exclude don't know, refused, and blanks
response_order = [
    "Extremely likely",
    "Very likely",
    "Somewhat likely",
    "Not too likely",
    "Not at all likely"
]

vuln_long = vuln_long[
    vuln_long["Response"].isin(response_order)
]

# Count responses by age group
vuln_summary = (
    vuln_long
    .groupby(["Response", "Age group"])
    .size()
    .reset_index(name="Respondents")
)

# Calculate percentage within each age group
vuln_summary["Percentage"] = (
    vuln_summary["Respondents"]
    / vuln_summary.groupby("Age group")["Respondents"].transform("sum")
)

# Pivot for connecting the two age-group dots
wide = (
    vuln_summary
    .pivot(index="Response", columns="Age group", values="Percentage")
    .reset_index()
    .dropna(subset=list(age_cols.keys()))
)

# Chart order and colors
age_order = ["Adults 30 and under", "Adults 65 and older"]

y = alt.Y(
    "Response:N",
    title=None,
    sort=response_order,
    axis=alt.Axis(labelLimit=220)
)

# Connecting lines
rules = (
    alt.Chart(wide)
    .mark_rule(color="gray", strokeWidth=2)
    .encode(
        y=y,
        x=alt.X(
            "Adults 30 and under:Q",
            title="Share of responses",
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(format="%", tickCount=5)
        ),
        x2="Adults 65 and older:Q"
    )
)

# Dots
dots = (
    alt.Chart(vuln_summary)
    .mark_circle(size=100)
    .encode(
        y=y,
        x=alt.X(
            "Percentage:Q",
            title="Share of responses",
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(format="%", tickCount=5)
        ),
        color=alt.Color(
            "Age group:N",
            title="Age group",
            scale=alt.Scale(
                domain=age_order,
                range=["#8FFF7B", "#567EE4"]
            )
        ),
        tooltip=[
            alt.Tooltip("Response:N", title="Likelihood"),
            alt.Tooltip("Age group:N"),
            alt.Tooltip("Respondents:Q"),
            alt.Tooltip("Percentage:Q", format=".1%", title="Share")
        ]
    )
)

vuln_dumbbell = (
    (rules + dots)
    .properties(
        title="Perceived Scam Vulnerability by Age Group",
        width=500,
        height=300
    )
)

vuln_dumbbell

alt.LayerChart(...)

In [84]:
importance_col = (
    "JOURSOCIETY_W168. Overall, how important do you think "
    "journalists are to the well-being of society?"
)
age_col = "Age category"

# Handle numeric codes or text labels
importance_values = df[importance_col].astype(str).str.strip()

important_responses = [
    "1", "1.0", "2", "2.0",
    "Extremely important", "Very important"
]

# Exclude missing/refused age categories
valid_age_df = df[
    df[age_col].astype(str).str.strip().isin(
        ["1", "1.0", "2", "2.0", "3", "3.0", "4", "4.0",
         "18-29", "30-49", "50-64", "65+"]
    )
].copy()

valid_age_df["Important"] = (
    importance_values.loc[valid_age_df.index]
    .isin(important_responses)
)

# Percentage within each age group
age_order = ["18-29", "30-49", "50-64", "65+"]

# Map numeric age codes to labels if needed
age_map = {
    "1": "18-29", "1.0": "18-29",
    "2": "30-49", "2.0": "30-49",
    "3": "50-64", "3.0": "50-64",
    "4": "65+", "4.0": "65+"
}

valid_age_df["Age group"] = (
    valid_age_df[age_col].astype(str).str.strip().replace(age_map)
)

age_summary = (
    valid_age_df
    .groupby("Age group")
    .agg(
        Respondents=("Important", "size"),
        Important=("Important", "sum")
    )
    .reset_index()
)

age_summary["Percentage"] = (
    age_summary["Important"] / age_summary["Respondents"] * 100
)

age_summary["Age group"] = pd.Categorical(
    age_summary["Age group"],
    categories=age_order,
    ordered=True
)

age_summary = age_summary.sort_values("Age group")
age_summary

,Age group,Respondents,Important,Percentage
0,18-29,918,593,64.596950
1,30-49,3010,1855,61.627907
2,50-64,2550,1558,61.098039
3,65+,2882,1885,65.405968


In [85]:
age_chart = (
    alt.Chart(age_summary)
    .mark_bar()
    .encode(
        x=alt.X(
            "Age group:O",
            title="Age group",
            sort=age_order
        ),
        y=alt.Y(
    "Percentage:Q",
    title="Respondents who say journalists are extremely or very important (%)",
    scale=alt.Scale(domain=[0, 100]),
    axis=alt.Axis(format=".1f")
),
        tooltip=[
            alt.Tooltip("Age group:O"),
            alt.Tooltip("Important:Q", title="Extremely/very important count"),
            alt.Tooltip("Respondents:Q", title="Valid respondents"),
            alt.Tooltip("Percentage:Q", format=".1f", title="Percentage")
        ]
    )
    .properties(
        title="Importance of Journalists to Society by Age Group",
        width=500,
        height=350
    )
)

age_chart

alt.Chart(...)

In [86]:
import pandas as pd
import altair as alt

importance_col = (
    "JOURSOCIETY_W168. Overall, how important do you think "
    "journalists are to the well-being of society?"
)
confidence_col = "journalist_confidence"
age_col = "Age category"

age_order = ["18-29", "30-49", "50-64", "65+"]

chart_df = df[[age_col, importance_col, confidence_col, 'Wave 168 weight']].copy()

# # Map age categories if they are still numeric
# age_map = {
#     1: "18-29", 2: "30-49", 3: "50-64", 4: "65+",
#     "1": "18-29", "2": "30-49", "3": "50-64", "4": "65+"
# }
# chart_df["Age group"] = chart_df[age_col].replace(age_map)

# Define positive responses using the existing text labels
chart_df["Importance positive"] = chart_df[importance_col].isin([
    "Extremely important", "Very important"
])

chart_df["Confidence positive"] = chart_df[confidence_col].isin([
    "A great deal of confidence", "A fair amount of confidence"
])

# Exclude missing and don't know/refused responses from each denominator
importance_valid = chart_df[
    chart_df[importance_col].isin([
        "Extremely important", "Very important",
        "Somewhat important", "Not too important",
        "Not at all important"
    ])
]

confidence_valid = chart_df[
    chart_df[confidence_col].isin([
        "A great deal of confidence",
        "A fair amount of confidence",
        "Not too much confidence",
        "No confidence at all"
    ])
]

def summarize_positive_weighted(data, flag_col, measure, group_col=age_col, weight_col=weight_col):
    data = data.copy()
    data["_weighted_flag"] = data[flag_col] * data[weight_col]

    summary = (
        data.groupby(group_col)
        .agg(
            Positive=("_weighted_flag", "sum"),
            Total=(weight_col, "sum")
        )
        .reset_index()
    )
    summary["Percentage"] = summary["Positive"] / summary["Total"] * 100
    summary["Measure"] = measure
    return summary

importance_summary = summarize_positive_weighted(
    importance_valid, "Importance positive", "Importance", group_col=age_col
)
confidence_summary = summarize_positive_weighted(
    confidence_valid, "Confidence positive", "Confidence", group_col=age_col
)

# Importance extends left; confidence extends right
importance_summary["Plot percentage"] = -importance_summary["Percentage"]
confidence_summary["Plot percentage"] = confidence_summary["Percentage"]

plot_df = pd.concat(
    [importance_summary, confidence_summary],
    ignore_index=True
)

bars = (
    alt.Chart(plot_df)
    .mark_bar()
    .encode(
        y=alt.Y(
            "Age category:O",
            sort=age_order,
            title=None
        ),
        x=alt.X(
            "Plot percentage:Q",
            title="Percentage within age group",
            scale=alt.Scale(domain=[-100, 100]),
            axis=alt.Axis(
                values=[-100, -75, -50, -25, 0, 25, 50, 75, 100],
                labelExpr="abs(datum.value) + '%'"
            )
        ),
        color=alt.Color(
            "Measure:N",
            scale=alt.Scale(
                domain=["Importance", "Confidence"],
                range=["#4C78A8", "#F58518"]
            ),
            legend=alt.Legend(title="Measure")
        ),
        tooltip=[
            alt.Tooltip("Age category:N"),
            alt.Tooltip("Measure:N"),
            alt.Tooltip("Positive:Q", title="Positive responses"),
            alt.Tooltip("Total:Q", title="Valid responses"),
            alt.Tooltip("Percentage:Q", format=".1f", title="Percentage (%)")
        ]
    )
)

zero_line = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="black")
    .encode(x="x:Q")
)

butterfly_chart = (
    bars + zero_line
).properties(
    title="Importance of Journalists vs. Confidence in Journalists by Age",
    width=600,
    height=300
)

butterfly_chart

alt.LayerChart(...)

In [87]:
import pandas as pd
import altair as alt

importance_col = (
    "JOURSOCIETY_W168. Overall, how important do you think "
    "journalists are to the well-being of society?"
)
confidence_col = "journalist_confidence"
party_col = "Party"

chart_df = df[[party_col, importance_col, confidence_col, "Wave 168 weight"]].copy()
chart_df["Party group"] = chart_df[party_col].astype(str).str.strip()

# Positive responses
chart_df["Importance positive"] = chart_df[importance_col].isin([
    "Extremely important", "Very important"
])

chart_df["Confidence positive"] = chart_df[confidence_col].isin([
    "A great deal of confidence", "A fair amount of confidence"
])

# Valid responses for each question (exclude missing and don't know/refused)
importance_valid = chart_df[
    chart_df[importance_col].isin([
        "Extremely important", "Very important",
        "Somewhat important", "Not too important",
        "Not at all important"
    ])
].copy()

confidence_valid = chart_df[
    chart_df[confidence_col].isin([
        "A great deal of confidence",
        "A fair amount of confidence",
        "Not too much confidence",
        "No confidence at all"
    ])
].copy()

def summarize_positive_weighted(data, flag_col, measure, group_col=party_col, weight_col=weight_col):
    data = data.copy()
    data["_weighted_flag"] = data[flag_col] * data[weight_col]

    summary = (
        data.groupby(group_col)
        .agg(
            Positive=("_weighted_flag", "sum"),
            Total=(weight_col, "sum")
        )
        .reset_index()
    )
    summary["Percentage"] = summary["Positive"] / summary["Total"] * 100
    summary["Measure"] = measure
    return summary

importance_summary = summarize_positive_weighted(
    importance_valid, "Importance positive", "Importance", group_col=party_col
)
confidence_summary = summarize_positive_weighted(
    confidence_valid, "Confidence positive", "Confidence", group_col=party_col
)

# Importance extends left; confidence extends right
importance_summary["Plot percentage"] = -importance_summary["Percentage"]
confidence_summary["Plot percentage"] = confidence_summary["Percentage"]

plot_df = pd.concat(
    [importance_summary, confidence_summary],
    ignore_index=True
)

party_order = (
    chart_df["Party group"]
    .value_counts()
    .index
    .tolist()
)

bars = (
    alt.Chart(plot_df)
    .mark_bar()
    .encode(
        y=alt.Y(
            "Party:N",
            sort=party_order,
            title="Political party"
        ),
        x=alt.X(
            "Plot percentage:Q",
            title="Percentage within political party",
            scale=alt.Scale(domain=[-100, 100]),
            axis=alt.Axis(
                values=[-100, -75, -50, -25, 0, 25, 50, 75, 100],
                labelExpr="abs(datum.value) + '%'"
            )
        ),
        color=alt.Color(
            "Measure:N",
            scale=alt.Scale(
                domain=["Importance", "Confidence"],
                range=["#4C78A8", "#F58518"]
            ),
            legend=alt.Legend(title="Measure")
        ),
        tooltip=[
            alt.Tooltip("Party:N", title="Political party"),
            alt.Tooltip("Measure:N"),
            alt.Tooltip("Positive:Q", title="Positive responses"),
            alt.Tooltip("Total:Q", title="Valid responses"),
            alt.Tooltip("Percentage:Q", format=".1f", title="Percentage (%)")
        ]
    )
)

zero_line = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="black")
    .encode(x="x:Q")
)

butterfly_chart = (
    bars + zero_line
).properties(
    title="Importance of Journalists vs. Confidence in Journalists by Political Party",
    width=600,
    height=max(250, len(party_order) * 55)
)

butterfly_chart

alt.LayerChart(...)

In [88]:
print("Unweighted:", chart_df.groupby(by = "Party")["Importance positive"].mean())
print("Weighted:", summarize_positive_weighted(importance_valid, "Importance positive", "x", "Party")[["Party", "Percentage"]])

Unweighted: Party
Democrat          0.807358
Independent       0.633475
Refused           0.515789
Republican        0.457681
Something else    0.577802
Name: Importance positive, dtype: float64
Weighted:             Party  Percentage
0        Democrat   76.389039
1     Independent   60.700725
2         Refused   57.585201
3      Republican   44.193762
4  Something else   54.811248


In [89]:
confidence_counts = (
    df.groupby(confidence_col, dropna=False)[weight_col]
    .sum()
    .reindex(confidence_order, fill_value=0)
    .rename_axis("Confidence")
    .reset_index(name="Weighted_n")
)


confidence_chart = (
    alt.Chart(confidence_counts)
    .mark_bar()
    .encode(
        x=alt.X(
            "Confidence:N",
            sort=confidence_order,
            title="Confidence in journalists",
            axis=alt.Axis(labelAngle=-35)
        ),
        y=alt.Y("Weighted_n:Q", title="Number of respondents"),
        tooltip=[
            alt.Tooltip("Confidence:N"),
            alt.Tooltip("Weighted_n:Q", format=",")
        ]
    )
    .properties(
        title="Confidence in Journalists",
        width=500,
        height=350
    )
)

confidence_chart

alt.Chart(...)

In [90]:
confidence_chart = (
    alt.Chart(confidence_counts)
    .mark_arc()
    .encode(
        theta=alt.Theta("Weighted_n:Q", title="Weighted respondents"),
        color=alt.Color(
            "Confidence:N",
            title="Confidence in journalists",
            sort=confidence_order,
            scale=alt.Scale(
                domain=confidence_order,
                scheme='oranges'
            )
        ),
        tooltip=[
            alt.Tooltip("Confidence:N", title="Confidence"),
            alt.Tooltip("Weighted_n:Q", title="Weighted respondents", format=",")
        ]
    )
    .properties(
        title="Confidence in Journalists",
        width=500,
        height=350
    )
)

confidence_chart

alt.Chart(...)

In [91]:
importance_col = (
    "JOURSOCIETY_W168. Overall, how important do you think "
    "journalists are to the well-being of society?"
)

importance_order = [
    "Extremely important",
    "Very important",
    "Somewhat important",
    "Not too important",
    "Not at all important",
    "Don't know/Refused/Web blank"
]

importance_counts = (
    df[importance_col]
    .value_counts()
    .reindex(importance_order, fill_value=0)
    .rename_axis("Importance")
    .reset_index(name="Count")
)

importance_chart = (
    alt.Chart(importance_counts)
    .mark_bar()
    .encode(
        x=alt.X(
            "Importance:N",
            sort=importance_order,
            title="Importance of journalists to society",
            axis=alt.Axis(labelAngle=-35)
        ),
        y=alt.Y("Count:Q", title="Number of respondents"),
        tooltip=[
            alt.Tooltip("Importance:N"),
            alt.Tooltip("Count:Q", format=",")
        ]
    )
    .properties(
        title="Importance of Journalists to Society",
        width=500,
        height=350
    )
)

importance_chart

alt.Chart(...)

In [92]:
import pandas as pd
import altair as alt

confidence_col = "journalist_confidence"
news_col = "NEWSJOUR_W168. As far as you know, does your news mostly come from..."

# Adjust this if your news column has a different exact name:
# print([c for c in df.columns if "NEWSJOUR_W168" in c])

# Group confidence responses
high_confidence = [
    "A great deal of confidence",
    "A fair amount of confidence"
]
low_confidence = [
    "Not too much confidence",
    "No confidence at all"
]

news_order = [
    "People you think are journalists",
    "Roughly an even split",
    "People you do not think are journalists",
    "Not sure",
    "Refused/Web blank"
]

chart_df = df[[confidence_col, news_col]].copy()

chart_df["Confidence group"] = chart_df[confidence_col].map(
    lambda x: (
        "Higher confidence" if x in high_confidence
        else "Lower confidence" if x in low_confidence
        else None
    )
)

# Keep only the two confidence groups and valid news-source responses
chart_df = chart_df[
    chart_df["Confidence group"].notna()
    & chart_df[news_col].isin(news_order)
].copy()

# Calculate response percentages within each confidence group
summary = (
    chart_df.groupby(["Confidence group", news_col])
    .size()
    .rename("Count")
    .reset_index()
    .rename(columns={news_col: "News source"})
)

summary["Percentage"] = (
    summary["Count"]
    / summary.groupby("Confidence group")["Count"].transform("sum")
    * 100
)

# Pivot so each news-source category has a percentage for both groups
wide = summary.pivot(
    index="News source",
    columns="Confidence group",
    values="Percentage"
).reset_index()

wide = wide.rename_axis(columns=None)

for group in ["Higher confidence", "Lower confidence"]:
    if group not in wide.columns:
        wide[group] = 0

wide = wide.fillna(0)
wide["Difference"] = (
    wide["Higher confidence"] - wide["Lower confidence"]
)
wide["News source"] = pd.Categorical(
    wide["News source"],
    categories=news_order,
    ordered=True
)
wide = wide.sort_values("News source")

# Dumbbell connecting lines
lines = (
    alt.Chart(wide)
    .mark_rule(color="gray", strokeWidth=2)
    .encode(
        y=alt.Y(
            "News source:N",
            sort=news_order,
            title=None
        ),
        x=alt.X(
            "Lower confidence:Q",
            title="Respondents (%)",
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(format=".0f")
        ),
        x2="Higher confidence:Q",
        tooltip=[
            alt.Tooltip("News source:N"),
            alt.Tooltip("Lower confidence:Q", format=".1f"),
            alt.Tooltip("Higher confidence:Q", format=".1f"),
            alt.Tooltip("Difference:Q", format=".1f")
        ]
    )
)

# Dots for each group
dots = (
    alt.Chart(wide)
    .transform_fold(
        ["Higher confidence", "Lower confidence"],
        as_=["Confidence group", "Percentage"]
    )
    .mark_circle(size=130)
    .encode(
        y=alt.Y("News source:N", sort=news_order, title=None),
        x=alt.X(
            "Percentage:Q",
            title="Respondents (%)",
            scale=alt.Scale(domain=[0, 100])
        ),
        color=alt.Color(
            "Confidence group:N",
            scale=alt.Scale(
                domain=["Higher confidence", "Lower confidence"],
                range=["#F554BC", "#840EA4"]
            ),
            title="Journalist confidence"
        ),
        tooltip=[
            alt.Tooltip("News source:N"),
            alt.Tooltip("Confidence group:N"),
            alt.Tooltip("Percentage:Q", format=".1f", title="Percentage (%)")
        ]
    )
)

dumbbell_chart = (
    (lines + dots)
    .properties(
        title="Where News Comes From by Confidence in Journalists",
        width=600,
        height=300
    )
)

dumbbell_chart

alt.LayerChart(...)

In [93]:
import pandas as pd
import altair as alt

importance_col = (
    "JOURSOCIETY_W168. Overall, how important do you think "
    "journalists are to the well-being of society?"
)
training_col = "JOURTRAINING_W168. Which of the following comes closer to your view, even if neither is exactly right?"

# If the column names differ, check:
# print([c for c in df.columns if "JOURTRAINING_W168" in c])

training_groups = [
    "People should have to go through formal training to be a journalist",
    "People should not have to go through formal training to be a journalist"
]

importance_order = [
    "Extremely important",
    "Very important",
    "Somewhat important",
    "Not too important",
    "Not at all important"
]

chart_df = df[[training_col, importance_col]].copy()

# Keep the two training-opinion groups and valid importance responses
chart_df = chart_df[
    chart_df[training_col].isin(training_groups)
    & chart_df[importance_col].isin(importance_order)
].copy()

# Count each importance response within each training group
summary = (
    chart_df.groupby([training_col, importance_col])
    .size()
    .rename("Count")
    .reset_index()
    .rename(columns={
        training_col: "Training opinion",
        importance_col: "Importance level"
    })
)

# Calculate percentages within each training group
summary["Percentage"] = (
    summary["Count"]
    / summary.groupby("Training opinion")["Count"].transform("sum")
    * 100
)

# Put the two groups side by side for the dumbbell chart
wide = summary.pivot(
    index="Importance level",
    columns="Training opinion",
    values="Percentage"
).reset_index()

wide.columns.name = None
wide = wide.fillna(0)

formal_group = training_groups[0]
no_formal_group = training_groups[1]

wide = wide.rename(columns={
    formal_group: "Should have formal training",
    no_formal_group: "Should not have formal training"
})

for col in ["Should have formal training", "Should not have formal training"]:
    if col not in wide.columns:
        wide[col] = 0

wide["Importance level"] = pd.Categorical(
    wide["Importance level"],
    categories=importance_order,
    ordered=True
)
wide = wide.sort_values("Importance level")

# Connecting lines
lines = (
    alt.Chart(wide)
    .mark_rule(color="gray", strokeWidth=2)
    .encode(
        y=alt.Y("Importance level:N", sort=importance_order, title=None),
        x=alt.X(
            "Should not have formal training:Q",
            title="Respondents (%)",
            scale=alt.Scale(domain=[0, 100])
        ),
        x2="Should have formal training:Q",
        tooltip=[
            alt.Tooltip("Importance level:N"),
            alt.Tooltip(
                "Should have formal training:Q",
                format=".1f",
                title="Should have formal training (%)"
            ),
            alt.Tooltip(
                "Should not have formal training:Q",
                format=".1f",
                title="Should not have formal training (%)"
            )
        ]
    )
)

# Dots
dots = (
    alt.Chart(wide)
    .transform_fold(
        [
            "Should have formal training",
            "Should not have formal training"
        ],
        as_=["Training opinion", "Percentage"]
    )
    .mark_circle(size=130)
    .encode(
        y=alt.Y("Importance level:N", sort=importance_order, title=None),
        x=alt.X(
            "Percentage:Q",
            title="Respondents (%)",
            scale=alt.Scale(domain=[0, 100])
        ),
        color=alt.Color(
            "Training opinion:N",
            scale=alt.Scale(
                domain=[
                    "Should have formal training",
                    "Should not have formal training"
                ],
                range=["#4C78A8", "#F58518"]
            ),
            legend=alt.Legend(title="View on journalist training")
        ),
        tooltip=[
            alt.Tooltip("Importance level:N"),
            alt.Tooltip("Training opinion:N"),
            alt.Tooltip("Percentage:Q", format=".1f", title="Percentage (%)")
        ]
    )
)

dumbbell_chart = (
    (lines + dots)
    .properties(
        title="Importance of Journalists by Views on Formal Training",
        width=600,
        height=300
    )
)

dumbbell_chart

alt.LayerChart(...)

In [94]:
characteristic_cols = {
    "Intelligent": "JOURCHAR_INT_W168. Intelligent // In general, how would you say each of the following describes most journalists?",
    "Relatable": "JOURCHAR_REL_W168. Relatable // In general, how would you say each of the following describes most journalists?",
    "Dishonest": "JOURCHAR_DISH_W168. Dishonest // In general, how would you say each of the following describes most journalists?",
    "Elitist": "JOURCHAR_ELITE_W168. Elitist // In general, how would you say each of the following describes most journalists?",
    "Biased": "JOURCHAR_BIAS_W168. Biased // In general, how would you say each of the following describes most journalists?",
    "Well-intentioned": "JOURCHAR_WELL_W168. Well-intentioned // In general, how would you say each of the following describes most journalists?"
}

response_order = [
    "Describes most journalists well",
    "Does not describe most journalists well",
    "Don't know/Refused/Web blank"
]

# Pull directly from the main df and reshape
long_df = df[list(characteristic_cols.values())].rename(
    columns={v: k for k, v in characteristic_cols.items()}
).melt(
    var_name="Characteristic",
    value_name="Response"
)

# Count each response for each characteristic
counts = (
    long_df.groupby(["Characteristic", "Response"], observed=True)
    .size()
    .reset_index(name="Count")
)

counts["Response"] = pd.Categorical(
    counts["Response"],
    categories=response_order,
    ordered=True
)

characteristic_order = list(characteristic_cols.keys())

chart = (
    alt.Chart(counts)
    .mark_bar()
    .encode(
        x=alt.X(
            "Characteristic:N",
            sort=characteristic_order,
            title="Journalist characteristic",
            axis=alt.Axis(labelAngle=-35)
        ),
        y=alt.Y("Count:Q", title="Number of respondents"),
        xOffset=alt.XOffset("Response:N", sort=response_order),
        color=alt.Color(
            "Response:N",
            scale=alt.Scale(
                domain=response_order,
                range=["#4C78A8", "#F58518", "#B8B8B8"]
            ),
            legend=alt.Legend(title="Response")
        ),
        tooltip=[
            alt.Tooltip("Characteristic:N"),
            alt.Tooltip("Response:N"),
            alt.Tooltip("Count:Q", format=",")
        ]
    )
    .properties(
        title="Perceptions of Journalists' Characteristics",
        width=600,
        height=400
    )
)

chart

alt.Chart(...)

In [95]:
weight_col = "Wave 168 weight"

valid_responses = [
    "Describes most journalists well",
    "Does not describe most journalists well",
]

# 1. Carry the weight through the melt
long_df = (
    df[["Party", weight_col, *characteristic_cols.values()]]
    .rename(columns={v: k for k, v in characteristic_cols.items()})
    .melt(
        id_vars=["Party", weight_col],
        var_name="Characteristic",
        value_name="Response",
    )
)

# 2. Keep only valid answers so the denominator excludes DK/Refused
valid_df = long_df[long_df["Response"].isin(valid_responses)].copy()

# Flag "describes well" and weight it
valid_df["Describes well"] = valid_df["Response"].eq("Describes most journalists well")
valid_df["Weighted flag"] = valid_df["Describes well"] * valid_df[weight_col]

# 3. Weighted share within each Characteristic x Party
well_share = (
    valid_df
    .groupby(["Characteristic", "Party"], observed=True)
    .agg(
        Weighted_yes=("Weighted flag", "sum"),
        Weighted_total=(weight_col, "sum"),
        Unweighted_n=("Describes well", "size"),
    )
    .reset_index()
)

well_share["Share"] = well_share["Weighted_yes"] / well_share["Weighted_total"]

# 4. Optional: drop the tiny "Refused" party group (n is too small to read)
well_share = well_share[well_share["Party"] != "Refused"]

# 5. Optional: focus on the on-theme traits
focus_traits = list(characteristic_cols)        # or list(characteristic_cols) for all six
plot_df = well_share[well_share["Characteristic"].isin(focus_traits)]

party_order_plot = ["Democrat", "Republican", "Independent", "Something else"]
party_colors = alt.Scale(
    domain=party_order_plot,
    range=["#4386D6", "#FF463D", "#54A24B", "#B279A2"],
)

chart = (
    alt.Chart(plot_df)
    .mark_bar()
    .encode(
        x=alt.X(
            "Characteristic:N",
            sort=focus_traits,
            title=None,
            axis=alt.Axis(labelAngle=0),
        ),
        xOffset=alt.XOffset("Party:N", sort=party_order_plot),
        y=alt.Y(
            "Share:Q",
            title="Share saying it describes most journalists well",
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(format="%"),
        ),
        color=alt.Color(
            "Party:N",
            title="Political party",
            sort=party_order_plot,
            scale=party_colors,
        ),
        tooltip=[
            alt.Tooltip("Characteristic:N"),
            alt.Tooltip("Party:N"),
            alt.Tooltip("Share:Q", format=".1%", title="Weighted share"),
            alt.Tooltip("Unweighted_n:Q", format=",", title="Respondents (unweighted)"),
        ],
    )
    .properties(
        title="Share of Believed Journalist Characteristics, by Party",
        width=450,
        height=350,
    )
)

chart

alt.Chart(...)

In [96]:
journalist_type_cols = {
    "Newspaper/news website writer":
        "ISJOURMED_PAP_W168. Writes for a newspaper or news website // Would you consider each of the following to be a journalist? Someone who...",
    "TV news reporter/host":
        "ISJOURMED_TV_W168. Reports on or hosts a TV news show // Would you consider each of the following to be a journalist? Someone who...",
    "Radio news reporter/host":
        "ISJOURMED_RAP_W168. Reports on or hosts a radio news show // Would you consider each of the following to be a journalist? Someone who...",
    "News podcast reporter/host":
        "ISJOURMED_POD_W168. Reports on or hosts a news podcast // Would you consider each of the following to be a journalist? Someone who...",
    "News newsletter writer":
        "ISJOURMED_LET_W168. Writes their own newsletter about news // Would you consider each of the following to be a journalist? Someone who...",
    "News creator on social media":
        "ISJOURMED_SM_W168. Makes their own videos or posts about news on social media // Would you consider each of the following to be a journalist? Someone who...",
    "Reports on current events":
        "ISJOURCONT_NEW_W168. Conducts their own reporting on current events // Would you consider each of the following to be a journalist? Someone who primarily...",
    "Comments on current events":
        "ISJOURCONT_OPIN_W168. Offers opinions or commentary on current events // Would you consider each of the following to be a journalist? Someone who primarily...",
    "Shares others' reporting":
        "ISJOURCONT_AGG_W168. Compiles and shares other people's reporting on current events // Would you consider each of the following to be a journalist? Someone who primarily..."
}

age_col = "Age category"
age_order = ["18-29", "30-49", "50-64", "65+"]
journalist_type_order = list(journalist_type_cols.keys())

age_map = {
    "1": "18-29", "1.0": "18-29",
    "2": "30-49", "2.0": "30-49",
    "3": "50-64", "3.0": "50-64",
    "4": "65+",   "4.0": "65+"
}

long_df = (
    df[[age_col, *journalist_type_cols.values()]]
    .rename(columns={v: k for k, v in journalist_type_cols.items()})
    .melt(
        id_vars=age_col,
        var_name="Journalist type",
        value_name="Response"
    )
    .rename(columns={age_col: "Age group"})
)

long_df["Age group"] = (
    long_df["Age group"].astype(str).str.strip().replace(age_map)
)

# Keep Yes responses; each bar shows the age mix among respondents who said Yes
yes_by_age = (
    long_df[long_df["Response"] == "Yes"]
    .groupby(["Journalist type", "Age group"], observed=True)
    .size()
    .reset_index(name="Respondents")
)

chart = (
    alt.Chart(yes_by_age)
    .mark_bar()
    .encode(
        x=alt.X(
            "Journalist type:N",
            title="Who is considered a journalist",
            sort=journalist_type_order,
            axis=alt.Axis(labelAngle=-45, labelLimit=180)
        ),
        y=alt.Y(
            "Respondents:Q",
            title="Share of “Yes” responses",
            stack="normalize",
            axis=alt.Axis(format="%")
        ),
        color=alt.Color(
            "Age group:N",
            title="Age group",
            sort=age_order,
            scale=alt.Scale(
                domain=age_order,
                range=["#97F1AD", "#59D077", "#26A947", "#098B29"]
            )
        ),
        order=alt.Order("Age group:N", sort="ascending"),
        tooltip=[
            alt.Tooltip("Journalist type:N", title="Journalist type"),
            alt.Tooltip("Age group:N"),
            alt.Tooltip("Respondents:Q", format=",")
        ]
    )
    .properties(
        title="Age Composition of Respondents Who Consider Each Type a Journalist",
        width=650,
        height=400
    )
)

chart

alt.Chart(...)

In [97]:
journalist_type_cols = {
    "Newspaper/news website writer":
        "ISJOURMED_PAP_W168. Writes for a newspaper or news website // Would you consider each of the following to be a journalist? Someone who...",
    "TV news reporter/host":
        "ISJOURMED_TV_W168. Reports on or hosts a TV news show // Would you consider each of the following to be a journalist? Someone who...",
    "Radio news reporter/host":
        "ISJOURMED_RAP_W168. Reports on or hosts a radio news show // Would you consider each of the following to be a journalist? Someone who...",
    "News podcast reporter/host":
        "ISJOURMED_POD_W168. Reports on or hosts a news podcast // Would you consider each of the following to be a journalist? Someone who...",
    "News newsletter writer":
        "ISJOURMED_LET_W168. Writes their own newsletter about news // Would you consider each of the following to be a journalist? Someone who...",
    "News creator on social media":
        "ISJOURMED_SM_W168. Makes their own videos or posts about news on social media // Would you consider each of the following to be a journalist? Someone who...",
    "Reports on current events":
        "ISJOURCONT_NEW_W168. Conducts their own reporting on current events // Would you consider each of the following to be a journalist? Someone who primarily...",
    "Comments on current events":
        "ISJOURCONT_OPIN_W168. Offers opinions or commentary on current events // Would you consider each of the following to be a journalist? Someone who primarily...",
    "Shares others' reporting":
        "ISJOURCONT_AGG_W168. Compiles and shares other people's reporting on current events // Would you consider each of the following to be a journalist? Someone who primarily..."
}

age_col = "Age category"
age_order = ["18-29", "30-49", "50-64", "65+"]
journalist_type_order = list(journalist_type_cols.keys())

age_map = {
    "1": "18-29", "1.0": "18-29",
    "2": "30-49", "2.0": "30-49",
    "3": "50-64", "3.0": "50-64",
    "4": "65+",   "4.0": "65+"
}

long_df = (
    df[[age_col, weight_col, *journalist_type_cols.values()]]
    .rename(columns={v: k for k, v in journalist_type_cols.items()})
    .melt(
        id_vars=[age_col, weight_col],
        var_name="Journalist type",
        value_name="Response"
    )
    .rename(columns={age_col: "Age group"})
)

long_df["Age group"] = (
    long_df["Age group"].astype(str).str.strip().replace(age_map)
)

# Keep valid Yes/No answers and the four age groups (drops DK/Refused)
valid_age = long_df[
    long_df["Response"].isin(["Yes", "No"])
    & long_df["Age group"].isin(age_order)
].copy()
valid_age["Weighted yes"] = valid_age[weight_col] * (valid_age["Response"] == "Yes")

# Willingness: weighted % of each age group that says "Yes, a journalist"
yes_by_age = (
    valid_age
    .groupby(["Journalist type", "Age group"], observed=True)
    .agg(Weighted_yes=("Weighted yes", "sum"),
         Weighted_total=(weight_col, "sum"),
         Unweighted_n=("Response", "size"))
    .reset_index()
)
yes_by_age["Share"] = yes_by_age["Weighted_yes"] / yes_by_age["Weighted_total"]

# Sort types by overall willingness (most accepted at the top)
_type_totals = valid_age.groupby("Journalist type")[["Weighted yes", weight_col]].sum()
journalist_type_order = (
    (_type_totals["Weighted yes"] / _type_totals[weight_col])
    .sort_values(ascending=False)
    .index.tolist()
)

# Range line from the lowest to the highest age group for each type
age_range = (
    yes_by_age.groupby("Journalist type")["Share"]
    .agg(Low="min", High="max")
    .reset_index()
)

age_axis_y = alt.Y(
    "Journalist type:N",
    title=None,
    sort=journalist_type_order,
    axis=alt.Axis(labelLimit=230)
)

age_lines = (
    alt.Chart(age_range)
    .mark_rule(color="gray", strokeWidth=2)
    .encode(
        y=age_axis_y,
        x=alt.X(
            "Low:Q",
            title="% of age group saying “Yes, a journalist”",
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(format="%")
        ),
        x2="High:Q"
    )
)

age_dots = (
    alt.Chart(yes_by_age)
    .mark_circle(size=110, opacity=1)
    .encode(
        y=age_axis_y,
        x=alt.X("Share:Q", scale=alt.Scale(domain=[0, 1])),
        color=alt.Color(
            "Age group:N",
            title="Age group",
            sort=age_order,
            scale=alt.Scale(
                domain=age_order,
                range=["#C9D1D9", "#8FA1B3", "#52667A", "#1F2D3D"]
            )
        ),
        tooltip=[
            alt.Tooltip("Journalist type:N", title="Journalist type"),
            alt.Tooltip("Age group:N"),
            alt.Tooltip("Share:Q", format=".1%", title="Weighted share saying yes"),
            alt.Tooltip("Unweighted_n:Q", format=",", title="Respondents (unweighted)")
        ]
    )
)

age_chart = age_lines + age_dots

final_age = age_chart.properties(
    title=alt.TitleParams(["Who counts as a journalist, by age group"], anchor="start"),
    width=380,
    height=300
)

final_age

alt.LayerChart(...)